In [2]:
import pandas as pd
import numpy as np
import datetime as dt

In [3]:
PATH = "/Users/elisalorandi/Desktop/case study_asset/EURUSD.csv"  

df = pd.read_csv(PATH)

# Timestamp: la colonna 'date' ha offset (es. -04:00), quindi usiamo utc=True
df["ts"] = pd.to_datetime(df["date"], utc=True)
df = df.sort_values("ts").reset_index(drop=True)

# Prezzo che useremo (close)
df["price"] = df["close"].astype(float)


In [4]:
# =========================
# 2) FILTER: periodo richiesto dalla consegna
# (1 Nov 2024 - 31 Oct 2025)
# =========================
start_all = pd.Timestamp("2024-11-01", tz="UTC")
end_all   = pd.Timestamp("2025-10-31 23:59:59", tz="UTC")

df = df[(df["ts"] >= start_all) & (df["ts"] <= end_all)].copy()
df["day"] = df["ts"].dt.date  # trading day = "giorno come appare nel dataset"


In [5]:
# =========================
# 3) SELECT: 60 trading days consecutivi
# =========================
trading_days = sorted(df["day"].unique())  # lista giorni (come da dataset)

# Scelta consigliata (pulita, evita fine anno): inizio Febbraio 2025
anchor_day = dt.date(2025, 2, 3)

if anchor_day not in trading_days:
    # fallback automatico: prendi il primo giorno disponibile >= anchor_day
    anchor_day = next(d for d in trading_days if d >= anchor_day)

anchor_idx = trading_days.index(anchor_day)
window_days = trading_days[anchor_idx : anchor_idx + 60]

if len(window_days) < 60:
    raise ValueError("Non ci sono abbastanza trading days dopo anchor_day per farne 60.")

df_60 = df[df["day"].isin(window_days)].copy()
df_60 = df_60.sort_values("ts").reset_index(drop=True)

print("60 trading days window:")
print("Start day:", window_days[0], "| End day:", window_days[-1], "| N days:", len(window_days))
print("Rows (minutes):", len(df_60))

60 trading days window:
Start day: 2025-02-03 | End day: 2025-04-13 | N days: 60
Rows (minutes): 58230


In [6]:
# =========================
# 4) VINCOLO: time-in-market >= 80%
# pos_t in {-1,0,+1}
# =========================
def time_in_market_checks(position: pd.Series,
                          day: pd.Series,
                          threshold: float = 0.80,
                          require_daily: bool = True) -> dict:
    """
    position: pd.Series con valori -1, 0, +1 (stessa lunghezza di df_60)
    day:      pd.Series con la data (df_60['day'])
    threshold: es. 0.80
    require_daily: se True controlla anche ogni singolo trading day
    """
    invested = (position != 0)

    tim_overall = invested.mean()

    out = {"tim_overall": float(tim_overall), "overall_pass": bool(tim_overall >= threshold)}

    if require_daily:
        tim_daily = invested.groupby(day).mean()
        out["tim_daily_min"] = float(tim_daily.min())
        out["daily_pass_all"] = bool((tim_daily >= threshold).all())
        out["tim_daily"] = tim_daily  # utile per debug/report
    return out

Simple Moving Average - 1min

In [7]:
# =========================
# PARAMETRI STRATEGIA
# =========================
SHORT_W = 30
LONG_W  = 90

# =========================
# 1) INDICATORI: SMA
# =========================
df_60["sma_s"] = df_60["price"].rolling(SHORT_W, min_periods=SHORT_W).mean()
df_60["sma_l"] = df_60["price"].rolling(LONG_W,  min_periods=LONG_W).mean()

# segnali "raw" (attenzione: qui è ancora contemporaneo al prezzo t)
# +1 se sma_s > sma_l, -1 altrimenti (sempre investito quando entrambe pronte)
df_60["pos_raw"] = np.where(df_60["sma_s"] > df_60["sma_l"], 1, -1)

# =========================
# 2) NO LOOK-AHEAD: shift della posizione
# La posizione di t si applica al return t -> t+1
# =========================
df_60["pos"] = df_60["pos_raw"].shift(1)

# prima che LONG_W sia pronto, vogliamo posizione = 0 (flat) perché non abbiamo segnale valido
df_60.loc[df_60["sma_l"].isna(), "pos"] = 0
df_60["pos"] = df_60["pos"].fillna(0).astype(int)

# =========================
# 3) VINCOLO time-in-market >= 80%
# (overall + per-day)
# =========================
checks = time_in_market_checks(df_60["pos"], df_60["day"], threshold=0.80, require_daily=True)

print("TiM overall:", round(checks["tim_overall"], 4), "| PASS:", checks["overall_pass"])
print("TiM daily min:", round(checks["tim_daily_min"], 4), "| PASS all days:", checks["daily_pass_all"])

if not (checks["overall_pass"] and checks["daily_pass_all"]):
    raise ValueError("Vincolo time-in-market NON rispettato (>=80%).")

# =========================
# 4) RETURNS: log-returns + strategy returns
# =========================
df_60["ret"] = np.log(df_60["price"]).diff()          # r_{t} = log(P_t) - log(P_{t-1})
df_60["ret_strat"] = df_60["pos"] * df_60["ret"]     # posizione applicata al return

# pulizia NaN iniziali
df_bt = df_60.dropna(subset=["ret", "ret_strat"]).copy()

# =========================
# 5) METRICHE: equity curve, Sharpe annualizzato, max drawdown
# =========================
# equity curve (log cum return -> exp)
df_bt["equity"] = np.exp(df_bt["ret_strat"].cumsum())

# max drawdown
roll_max = df_bt["equity"].cummax()
drawdown = df_bt["equity"] / roll_max - 1.0
max_dd = drawdown.min()

# Sharpe annualizzato: su 1-min devi annualizzare con attenzione
# - per FX 24h: ~ 252 giorni * 1440 min = 362,880 min/anno
# - però il dataset potrebbe non avere esattamente 1440 min al giorno (weekend, missing)
#   quindi stimiamo i minuti/anno come: mean minutes/day * 252
minutes_per_day = df_bt.groupby("day").size().mean()
ann_factor = np.sqrt(minutes_per_day * 252)

mu = df_bt["ret_strat"].mean()
sigma = df_bt["ret_strat"].std(ddof=0)

sharpe_ann = (mu / sigma) * ann_factor if sigma > 0 else np.nan

print("\n--- SMA Crossover Results (in-sample 60 trading days) ---")
print("Short/Long:", SHORT_W, "/", LONG_W)
print("Mean ret (per min):", mu)
print("Std  ret (per min):", sigma)
print("Sharpe annualized:", sharpe_ann)
print("Max drawdown:", max_dd)
print("Final equity:", df_bt["equity"].iloc[-1])


TiM overall: 0.9985 | PASS: True
TiM daily min: 0.9176 | PASS all days: True

--- SMA Crossover Results (in-sample 60 trading days) ---
Short/Long: 30 / 90
Mean ret (per min): 1.7124735310971495e-06
Std  ret (per min): 0.00019427057219484738
Sharpe annualized: 4.359243807549835
Max drawdown: -0.024974834164625537
Final equity: 1.1048566756269267
